In [4]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [6]:
loader = TextLoader('langchain_rag_dataset.txt')
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size= 200, chunk_overlap= 20)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent

In [7]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


C:\Users\raavi\AppData\Local\Temp\ipykernel_31692\869657906.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
retriever = vectorstore.as_retriever(
    search_type ="mmr",
    search_kwargs ={"k":3}
)


In [14]:
PromptTemplate = PromptTemplate.from_template(
    """
    Answer the question based on the context provider
    context:
    {context}

    Question:{input}

    """
)
llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0)


In [15]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=PromptTemplate)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain= document_chain)

In [16]:
query ={
    "input" : "How does Langchain Support agents and memory?"
}
response = rag_chain.invoke(query)
response

{'input': 'How does Langchain Support agents and memory?',
 'context': [Document(id='7744cfa7-e13b-4326-aa86-4ab30b5cf781', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
  Document(id='3ea7cd51-3972-4530-93b6-169e5a3e3753', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.'),
  Document(id='d74b043e-298a-4928-a53e-0be092df2eac', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Chroma is a lightweight vector store often used in LangChain for embedding-based document storage and retrieval.')],
 'answer': 'LangChain supports agents and memory by providing abstractions for working with them. Specifically, it provides memory capabilities that help models retain previous interactions, making multi-